# Step 18 — Offline blend optimisation

**Settings: Accelerator = `None` (CPU), Internet = `Off`.** Costs nothing, spends no submissions.

**Inputs:** competition data, `rsna-knee-03-extract-all`, `rsna-knee-05-train-fold0`, `rsna-knee-08-train-folds`, `rsna-knee-14-train224-fold0`, `rsna-knee-16-train224-folds` (both versions if the picker allows)

## Why this is worth doing first

You have out-of-fold predictions from both resolutions: every training study has a prediction from the one model that never saw it. That's **~4,216 studies** to tune blending on — seventy times the gold holdout, and it costs no GPU and no submissions.

Four questions, all answerable here:

1. **Does adding the 160px models help?** They're weaker (gold 0.805 vs 0.815). Diversity might beat dilution, or might not.
2. **Mean or rank average?** AUC only cares about ordering. Rank-averaging removes scale mismatch between models, which sometimes matters more than the raw values.
3. **One blend weight, or one per label?** Some labels may prefer 224px strongly, others may benefit from the 160px view.
4. **How much headroom is left in blending at all?**

Anything that doesn't help here won't help on the leaderboard either — so this is the cheap filter before spending GPU hours.

### 1 · Load out-of-fold predictions from both resolutions

In [ ]:
import glob, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

ROOT   = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()
KEYS   = ["acl","mcl","medial_meniscus","lateral_meniscus","medial_oa","lateral_oa",
          "pf_oa","effusion","synovitis","bakers","contusion","fracture"]

def gather(pattern):
    fs = glob.glob(f"/kaggle/input/notebooks/*/*/{pattern}")
    if not fs: return None
    df = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
    return df.drop_duplicates("StudyInstanceUID")

o160 = gather("oof_fold*.csv")
o224 = gather("oof224_fold*.csv")
if o160 is None:
    o160 = gather("oof_all.csv")

print("160px OOF:", None if o160 is None else o160.shape)
print("224px OOF:", None if o224 is None else o224.shape)

pseudo = pd.read_csv(glob.glob("/kaggle/input/notebooks/*/*/pseudo_labels.csv")[0])

m = o160.merge(o224, on="StudyInstanceUID", suffixes=("_160", "_224"))
print("overlapping studies:", len(m))

Y = np.nan_to_num(
    pseudo.set_index("StudyInstanceUID").loc[m.StudyInstanceUID, KEYS].values / 100.0, nan=0.2)
Ybin = (Y > 0.5).astype(int)

P160 = m[[f"{l}_160" for l in LABELS]].values
P224 = m[[f"{l}_224" for l in LABELS]].values
print("P160", P160.shape, "P224", P224.shape)

### 2 · Baselines

In [ ]:
def macro(P):
    out = []
    for c in range(P.shape[1]):
        y = Ybin[:, c]
        if y.min() == y.max(): continue
        out.append(roc_auc_score(y, P[:, c]))
    return float(np.mean(out)), out

def per_label(P):
    return macro(P)[1]

a160, l160 = macro(P160)
a224, l224 = macro(P224)
print(f"160px OOF macro : {a160:.4f}")
print(f"224px OOF macro : {a224:.4f}")
print(f"\n{'label':<18}{'160':>8}{'224':>8}{'diff':>8}")
for l, x, y in zip(LABELS, l160, l224):
    print(f"{l:<18}{x:>8.3f}{y:>8.3f}{y-x:>+8.3f}")

### 3 · Question 1 & 2 — does blending help, and mean or rank?

**Rank averaging** replaces each model's raw scores with their rank order before averaging. Since AUC scores only the ordering, this discards nothing — and it stops a model with a wider output range from dominating the average.

In [ ]:
def to_rank(P):
    return np.column_stack([rankdata(P[:, c]) / len(P) for c in range(P.shape[1])])

R160, R224 = to_rank(P160), to_rank(P224)

print(f"{'w224':>6}{'mean':>10}{'rank':>10}")
best_mean = best_rank = (0, None)
for w in np.arange(0, 1.01, 0.1):
    am, _ = macro((1-w) * P160 + w * P224)
    ar, _ = macro((1-w) * R160 + w * R224)
    if am > best_mean[0]: best_mean = (am, w)
    if ar > best_rank[0]: best_rank = (ar, w)
    mark = "  <-- 224 only" if w == 1.0 else ""
    print(f"{w:>6.1f}{am:>10.4f}{ar:>10.4f}{mark}")

print(f"\nbest MEAN blend: w224={best_mean[1]:.1f} -> {best_mean[0]:.4f}")
print(f"best RANK blend: w224={best_rank[1]:.1f} -> {best_rank[0]:.4f}")
print(f"224px alone    : {a224:.4f}")
gain = max(best_mean[0], best_rank[0]) - a224
print(f"\nblending gain over 224 alone: {gain:+.4f}")
print("(under +0.002 -> not worth the extra inference time)")

### 4 · Question 3 — per-label blend weights

Some findings may genuinely prefer one resolution. Osteoarthritis is a large-scale texture change that might survive downsampling fine; a meniscal tear is small and should favour 224px.

**Caveat:** fitting twelve weights on the same data you evaluate on is mild overfitting. With 4,216 studies it's modest, but treat any gain here as optimistic and prefer the global weight unless the per-label version wins clearly.

In [ ]:
use_rank = best_rank[0] >= best_mean[0]
A, B = (R160, R224) if use_rank else (P160, P224)
print("using", "RANK" if use_rank else "MEAN", "averaging\n")

ws, rows = [], []
for c in range(12):
    y = Ybin[:, c]
    if y.min() == y.max():
        ws.append(1.0); rows.append((LABELS[c], np.nan, np.nan, 1.0)); continue
    best = (0, 1.0)
    for w in np.arange(0, 1.01, 0.05):
        a = roc_auc_score(y, (1-w) * A[:, c] + w * B[:, c])
        if a > best[0]: best = (a, w)
    ws.append(best[1])
    rows.append((LABELS[c], roc_auc_score(y, B[:, c]), best[0], best[1]))

print(f"{'label':<18}{'224 only':>10}{'blended':>10}{'w224':>7}{'gain':>8}")
for l, base, bl, w in rows:
    print(f"{l:<18}{base:>10.3f}{bl:>10.3f}{w:>7.2f}{bl-base:>+8.3f}")

Pbl = np.column_stack([(1-ws[c]) * A[:, c] + ws[c] * B[:, c] for c in range(12)])
apl, _ = macro(Pbl)
print(f"\nper-label blend : {apl:.4f}")
print(f"global blend    : {max(best_mean[0], best_rank[0]):.4f}")
print(f"224px alone     : {a224:.4f}")
print("\nBLEND_W =", [round(w, 2) for w in ws])

### 5 · Verdict

In [ ]:
cands = {
    "224px alone":        a224,
    "global blend":       max(best_mean[0], best_rank[0]),
    "per-label blend":    apl,
}
best_name = max(cands, key=cands.get)
print("="*54)
for k, v in sorted(cands.items(), key=lambda x: -x[1]):
    print(f"  {k:<20} {v:.4f}   {v - a224:+.4f}")
print("="*54)
print(f"WINNER: {best_name}")
print(f"averaging: {'RANK' if use_rank else 'MEAN'}")
print(f"global w224 = {best_rank[1] if use_rank else best_mean[1]:.2f}")
print()
if cands[best_name] - a224 < 0.002:
    print("Gain is negligible -- stay with 224px alone and spend the effort")
    print("on a second backbone instead.")
else:
    print("Worth implementing in the inference notebook.")